## Quy trình tiền xử lý dữ liệu hoa Iris

Mục tiêu của bài toán này chuẩn bị một bộ dữ liệu hoàn chỉnh, sạch sẽ và chuẩn hóa để đưa vào các thuật toán Machine Learning sau này. Quy trình bao gồm: Làm sạch dữ liệu rác, Trực quan hóa (EDA), Mã hóa nhãn (Label Encoding) và Chuẩn hóa (Scaling).

In [ ]:
# Cell 2: Import thư viện cần thiết
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
import glob

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Cell 3: [Xử lý Metadata] Quét file metadata và trích xuất tên cột động
dataset_dir = 'dataset/'
search_pattern = os.path.join(dataset_dir, '*iris*names*')
matched_files = glob.glob(search_pattern)

if not matched_files:
    raise FileNotFoundError("Không tìm thấy file metadata nào chứa chữ 'iris' và 'names'")

names_file_path = matched_files[0]
print(f"Đang đọc metadata từ: {names_file_path}")

extracted_names = []
with open(names_file_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

capture = False
for line in lines:
    if line.startswith('7. Attribute Information:'):
        capture = True
        continue
    if capture and line.startswith('8. Missing Attribute Values:'):
        break
        
    if capture:
        # Dùng Regex móc ngoặc số thứ tự đầu dòng (Vd: 1. sepal length in cm)
        match = re.search(r'\s*\d+\.\s+(.*)', line)
        if match:
            raw_name = match.group(1).strip().lower()
            # Chuẩn hóa tên cột có gạch nối để chạy code tiện nhất
            if 'sepal length' in raw_name: col = 'sepal-length'
            elif 'sepal width' in raw_name: col = 'sepal-width'
            elif 'petal length' in raw_name: col = 'petal-length'
            elif 'petal width' in raw_name: col = 'petal-width'
            elif 'class' in raw_name: col = 'class'
            else: col = raw_name
            extracted_names.append(col)

print("✅ Danh sách tên cột động đã trích xuất:", extracted_names)

In [ ]:
# Cell 4: [Load Data] Đọc dữ liệu CSV và hiển thị màn hình
data_path = 'dataset/iris.csv'
df = pd.read_csv(data_path, header=None, names=extracted_names)

print("5 dòng quan sát đầu tiên của dữ liệu hoa Iris:")
display(df.head())

## 1. Phân tích tính toàn vẹn và Thống kê mô tả

In [ ]:
# Cell 6: Thống kê mô tả và Dữ liệu trùng lặp
print("[INFO] Cấu trúc dữ liệu và Kiểu dữ liệu:")
df.info()

print("\n[DESCRIBE] Thống kê mô tả tập dữ liệu:")
display(df.describe())

# Kiểm tra và hiển thị các dòng trùng lặp (Duplicated Rows)
dups = df.duplicated()
print(f"\n[CẢNH BÁO] Phát hiện {dups.sum()} dòng dữ liệu bị trùng lặp âm thầm!")
if dups.sum() > 0:
    print("Chi tiết các dòng bị trùng lặp:")
    display(df[dups])

## 2. Trực quan hóa dữ liệu (EDA)

Giai đoạn này giúp ta có cái nhìn chuyên sâu để phát hiện cấu trúc quần thể từng loài hoa.

In [ ]:
# Cell 8: Trực quan hóa dữ liệu (Hình vẽ trực tiếp trên Jupyter)
sns.set_theme(style="whitegrid")

input_cols = extracted_names[:-1] # 4 tính chất kích thước hoa

# 1. Boxplot (Xem tổng quan sự lệch chuẩn và Outliers)
plt.figure(figsize=(10, 6))
sns.boxplot(data=df[input_cols], palette="Set2")
plt.title("Biểu đồ phân bố Đặc điểm sinh học (Boxplot)")
plt.show()

# 2. Histogram (Tần suất phân bổ)
df[input_cols].hist(figsize=(10, 8), bins=20, color="#3498db", edgecolor="black")
plt.suptitle("Phân phối Histogram của 4 chỉ số kích thước", y=1.02)
plt.tight_layout()
plt.show()

# 3. Pairplot (Ma trận tán xạ 2 chiều để nhìn ra sự tách biệt giống phân loại)
sns.pairplot(df, hue="class", palette="viridis", markers=["o", "s", "D"])
plt.suptitle("Ma trận tán xạ (Scatter Plot) theo cấu trúc Loài hoa", y=1.02)
plt.show()

## 3. Biến đổi dữ liệu (Data Transformation)

Loại bỏ dữ liệu rác, Chuyển đổi nhãn chuỗi chữ thành số học (Mã hóa) và Đưa các dải số học chênh lệch về không gian biến đổi chung (Chuẩn hóa Scaling).

In [ ]:
# Cell 10: [Mã hóa nhãn] Xóa dòng trùng và Label Encoding
df_clean = df.drop_duplicates(ignore_index=True)
print(f"Đã xóa các dòng trùng lặp. Kích thước mẫu còn lại: {df_clean.shape}\n")

le = LabelEncoder()
df_clean['class_encoded'] = le.fit_transform(df_clean['class'])

print("--- BẢNG ĐỐI CHIẾU MÃ HÓA NHÃN CỦA LOÀI HOA ---")
mapping = dict(zip(le.classes_, range(len(le.classes_))))
for flower_name, class_id in mapping.items():
    print(f"Loài '{flower_name}'  -->  Code: {class_id}")

print("\nDữ liệu sau khi chèn thêm cột Nhãn số học:")
display(df_clean[['class', 'class_encoded']].head(2))
display(df_clean[['class', 'class_encoded']].tail(2))

In [ ]:
# Cell 11: [Chuẩn hóa] Áp dụng StandardScaler
scaler = StandardScaler()
df_scaled = df_clean.copy()

df_scaled[input_cols] = scaler.fit_transform(df_clean[input_cols])
df_scaled = df_scaled.drop('class', axis=1) # Bỏ cột chuỗi chữ, giữ lại nhãn Encoder

print("5 dòng đầu của dữ liệu đã được Trừ Độ Lệch và Ép không gian Tỷ Lệ (Z-score):")
display(df_scaled.head())

## 4. Phân chia dữ liệu thực nghiệm

In [ ]:
# Cell 13: Cắt xẻ dữ liệu Train - Test (70/30)
X = df_scaled[input_cols].values
y = df_scaled['class_encoded'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=42, stratify=y)

print("Đã chia tách tỷ lệ 70/30 Train-Test thành công!")
print(f"🔹 [Input] Trọng lượng mẫu Train: {X_train.shape}")
print(f"🔹 [Input] Trọng lượng mẫu Test :  {X_test.shape}")
print(f"🔸 [Output] Số lượng đáp án Model Train: {y_train.shape}")
print(f"🔸 [Output] Số lượng đáp án Khảo thí Test:  {y_test.shape}")

## KẾT LUẬN
Dữ liệu đã được chải chuốt sạch sẽ, vượt qua bước khảo sát EDA tường minh, không lọt ngoại lai, cũng như hoàn toàn sẵn sàng cung cấp nhiên liệu để đưa vào các đường ống thuật toán luyện mô hình học thuật lâm sàng (Machine Learning Models).